# Notebook 03 - Camada Gold e Analytics

Neste notebook será construída a camada **Gold** da arquitetura Medallion.

A camada Gold contém dados agregados e preparados para análises e dashboards no Power BI.

Ao final, os arquivos serão exportados em **Parquet** e **CSV**, permitindo sua utilização em diferentes ferramentas analíticas.

In [1]:
import os

import pandas as pd

## Leitura da camada Silver


In [2]:
silver = pd.read_parquet(
    "../datalake/silver/silver.parquet"
)

print(f"Registros: {len(silver)}")

silver.head()

Registros: 23475


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,uop,_source_file,_batch_id,_ingested_at,nova_coluna,ano,mes,dia,total_vitimas,acidente_grave
0,753147,2026-02-18,QUARTA-FEIRA,13:40:00,RS,158,"565,9",SANTANA DO LIVRAMENTO,DESRESPEITAR A PREFERÊNCIA NO CRUZAMENTO,COLISÃO TRANSVERSAL,...,UOP01-DEL11-RS,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,2,18,2,False
1,758570,2026-03-15,DOMINGO,12:30:00,MG,40,701,BARBACENA,AUSÊNCIA DE REAÇÃO DO CONDUTOR,COLISÃO LATERAL MESMO SENTIDO,...,UOP02-DEL05-MG,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,3,15,1,False
2,747911,2026-01-24,SÁBADO,16:50:00,MG,381,"438,3",SANTA LUZIA,PISTA ESBURACADA,SAÍDA DE LEITO CARROÇÁVEL,...,UOP01-DEL01-MG,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,1,24,2,False
3,751057,2026-02-08,DOMINGO,20:45:00,MT,364,266,JACIARA,INGESTÃO DE ÁLCOOL PELO CONDUTOR,COLISÃO COM OBJETO,...,UOP01-DEL02-MT,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,2,8,1,False
4,762973,2026-04-05,DOMINGO,07:50:00,RS,448,22,PORTO ALEGRE,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,COLISÃO COM OBJETO,...,UOP01-DEL01-RS,../../data/datatran2026.csv,LOTE_1,2026-06-26 00:44:45.849037,NaN,2026,4,5,1,False


## Criação da pasta Gold

In [3]:
os.makedirs(
    "../datalake/gold",
    exist_ok=True
)

# Gold - Acidentes por Estado

Perguntas respondidas:

- Quais estados registram mais acidentes?
- Quais estados possuem mais mortos?
- Quais estados possuem mais feridos?

In [4]:
gold_uf = (
    silver
    .groupby("uf")
    .agg(
        acidentes=("id","count"),
        mortos=("mortos","sum"),
        feridos=("feridos","sum"),
        veiculos=("veiculos","sum")
    )
    .reset_index()
)

gold_uf

,uf,acidentes,mortos,feridos,veiculos
0,AC,94,7,118,171
1,AL,203,30,244,376
2,AM,44,11,53,98
3,AP,52,6,63,106
4,BA,1344,177,1798,2961
5,CE,343,45,422,704
6,DF,327,18,349,644
7,ES,763,48,991,1561
8,GO,1101,107,1225,2174
9,MA,431,77,435,928


In [5]:
gold_uf.to_parquet(
    "../datalake/gold/gold_uf.parquet",
    index=False
)

print("Arquivo salvo.")

Arquivo salvo.


# Gold - Evolução Mensal

Pergunta respondida:

**Como os acidentes evoluem ao longo do tempo?**

In [6]:
gold_mensal = (
    silver
    .groupby(
        ["ano","mes"]
    )
    .agg(
        acidentes=("id","count"),
        mortos=("mortos","sum"),
        feridos=("feridos","sum")
    )
    .reset_index()
)

gold_mensal

,ano,mes,acidentes,mortos,feridos
0,2026,1,5822,494,7210
1,2026,2,5591,445,6785
2,2026,3,6119,469,6717
3,2026,4,5943,510,6845


In [7]:
gold_mensal.to_parquet(
    "../datalake/gold/gold_mensal.parquet",
    index=False
)

# Gold - Causas dos Acidentes

Pergunta:

**Quais são as principais causas dos acidentes?**

In [8]:
gold_causa = (
    silver
    .groupby("causa_acidente")
    .agg(
        acidentes=("id","count"),
        mortos=("mortos","sum"),
        feridos=("feridos","sum")
    )
    .sort_values(
        "acidentes",
        ascending=False
    )
    .reset_index()
)

gold_causa

,causa_acidente,acidentes,mortos,feridos
0,AUSÊNCIA DE REAÇÃO DO CONDUTOR,3701,247,4170
1,REAÇÃO TARDIA OU INEFICIENTE DO CONDUTOR,3476,181,4021
2,ACESSAR A VIA SEM OBSERVAR A PRESENÇA DOS OUTR...,2270,133,2915
3,CONDUTOR DEIXOU DE MANTER DISTÂNCIA DO VEÍCULO...,1387,38,1594
4,VELOCIDADE INCOMPATÍVEL,1310,155,1712
...,...,...,...,...
63,DEIXAR DE ACIONAR O FAROL DA MOTOCICLETA (OU S...,2,0,4
64,FARÓIS DESREGULADOS,2,1,1
65,SINALIZAÇÃO ENCOBERTA,2,0,4
66,MODIFICAÇÃO PROIBIDA,1,0,2


In [9]:
gold_causa.to_parquet(
    "../datalake/gold/gold_causa.parquet",
    index=False
)

# Gold - Municípios

Pergunta:

**Quais municípios concentram mais acidentes?**

In [10]:
gold_municipio = (
    silver
    .groupby("municipio")
    .agg(
        acidentes=("id","count"),
        mortos=("mortos","sum")
    )
    .sort_values(
        "acidentes",
        ascending=False
    )
    .reset_index()
)

gold_municipio

,municipio,acidentes,mortos
0,BRASILIA,327,18
1,DUQUE DE CAXIAS,266,12
2,SAO JOSE,256,3
3,RECIFE,245,16
4,BETIM,232,6
...,...,...,...
1660,IRUPI,1,0
1661,IRAI,1,0
1662,CACAULANDIA,1,0
1663,CALDEIRAO GRANDE,1,1


In [11]:
gold_municipio.to_parquet(
    "../datalake/gold/gold_municipio.parquet",
    index=False
)

# Gold - Dia da Semana

Pergunta:

**Em quais dias ocorrem mais acidentes?**

In [12]:
gold_dia = (
    silver
    .groupby("dia_semana")
    .agg(
        acidentes=("id","count"),
        mortos=("mortos","sum"),
        feridos=("feridos","sum")
    )
    .reset_index()
)

gold_dia

,dia_semana,acidentes,mortos,feridos
0,DOMINGO,3657,342,4327
1,QUARTA-FEIRA,3093,229,3529
2,QUINTA-FEIRA,3396,247,3938
3,SEGUNDA-FEIRA,3168,265,3710
4,SEXTA-FEIRA,3578,280,4218
5,SÁBADO,3605,318,4261
6,TERÇA-FEIRA,2978,237,3574


In [13]:
gold_dia.to_parquet(
    "../datalake/gold/gold_dia_semana.parquet",
    index=False
)

# Gold - Tipo de Acidente

Pergunta:

**Quais tipos de acidentes são mais frequentes e mais letais?**

In [14]:
gold_tipo = (
    silver
    .groupby("tipo_acidente")
    .agg(
        acidentes=("id","count"),
        mortos=("mortos","sum"),
        feridos=("feridos","sum")
    )
    .sort_values(
        "acidentes",
        ascending=False
    )
    .reset_index()
)

gold_tipo

,tipo_acidente,acidentes,mortos,feridos
0,COLISÃO TRASEIRA,4643,223,5312
1,SAÍDA DE LEITO CARROÇÁVEL,3425,196,4080
2,COLISÃO TRANSVERSAL,3043,178,3964
3,COLISÃO LATERAL MESMO SENTIDO,2593,80,2936
4,TOMBAMENTO,2097,83,2487
5,COLISÃO COM OBJETO,1600,102,1621
6,COLISÃO FRONTAL,1456,604,2460
7,QUEDA DE OCUPANTE DE VEÍCULO,1153,38,1373
8,ATROPELAMENTO DE PEDESTRE,945,267,885
9,COLISÃO LATERAL SENTIDO OPOSTO,675,67,885


In [15]:
gold_tipo.to_parquet(
    "../datalake/gold/gold_tipo_acidente.parquet",
    index=False
)

# Gold - Condição Meteorológica

Pergunta:

**Como as condições meteorológicas influenciam os acidentes?**

In [16]:
gold_clima = (
    silver
    .groupby("condicao_metereologica")
    .agg(
        acidentes=("id","count"),
        mortos=("mortos","sum"),
        feridos=("feridos","sum")
    )
    .sort_values(
        "acidentes",
        ascending=False
    )
    .reset_index()
)

gold_clima

,condicao_metereologica,acidentes,mortos,feridos
0,CÉU CLARO,13900,1154,16003
1,NUBLADO,4156,347,4784
2,CHUVA,2854,239,3726
3,SOL,1278,72,1566
4,GAROA/CHUVISCO,846,57,971
5,IGNORADO,290,25,302
6,NEVOEIRO/NEBLINA,130,23,176
7,VENTO,20,1,29
8,NEVE,1,0,0


In [17]:
gold_clima.to_parquet(
    "../datalake/gold/gold_clima.parquet",
    index=False
)

# Conversão para CSV

Nesta etapa, todos os arquivos Parquet da camada Gold serão convertidos para CSV, facilitando a importação no Power BI.

In [18]:
pasta_gold = "../datalake/gold"

for arquivo in os.listdir(pasta_gold):

    if arquivo.endswith(".parquet"):

        caminho = os.path.join(
            pasta_gold,
            arquivo
        )

        df = pd.read_parquet(caminho)

        nome_csv = arquivo.replace(
            ".parquet",
            ".csv"
        )

        df.to_csv(
            os.path.join(
                pasta_gold,
                nome_csv
            ),
            index=False,
            sep=";",
            encoding="utf-8-sig"
        )

        print(f"{nome_csv} criado.")

gold_causa.csv criado.
gold_clima.csv criado.
gold_dia_semana.csv criado.
gold_mensal.csv criado.
gold_municipio.csv criado.
gold_tipo_acidente.csv criado.
gold_uf.csv criado.


# Conclusão

Neste notebook foi realizada a construção da camada Gold da arquitetura Medallion.